In [1]:

!pip install -q -U requests pydantic pandas pytest python-dotenv


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:

import os
import requests
import pandas as pd

from pydantic import BaseModel, Field
from typing import Optional
from google.colab import userdata

os.environ["OPENALEX_API_KEY"] = userdata.get(
    "OPENALEX_API_KEY"
)

print("Libraries imported successfully.")


Libraries imported successfully.


In [5]:

class Paper(BaseModel):

    paper_id: str = Field(min_length=1)
    title: str = Field(min_length=1)

    authors: list[str] = Field(default_factory=list)

    publication_year: Optional[int] = None
    doi: Optional[str] = None
    abstract: Optional[str] = None

    cited_by_count: int = Field(default=0, ge=0)

    url: Optional[str] = None
    pdf_url: Optional[str] = None


In [6]:

def reconstruct_abstract(inverted_index):

    if not inverted_index:
        return None

    words = {}

    for word, positions in inverted_index.items():

        for position in positions:
            words[position] = word

    abstract = " ".join(
        words[position]
        for position in sorted(words)
    )

    return abstract


In [7]:

example = {
    "Artificial": [0],
    "intelligence": [1],
    "improves": [2],
    "education": [3]
}

print(reconstruct_abstract(example))


Artificial intelligence improves education


In [8]:

def parse_paper(data: dict) -> Paper:

    authors = []

    for item in data.get("authorships", []):

        author = item.get("author") or {}
        name = author.get("display_name")

        if name:
            authors.append(name)

    location = data.get("primary_location") or {}
    best_oa = data.get("best_oa_location") or {}

    return Paper(
        paper_id=data.get("id") or "",
        title=data.get("title") or "",
        authors=authors,
        publication_year=data.get("publication_year"),
        doi=data.get("doi"),
        abstract=reconstruct_abstract(
            data.get("abstract_inverted_index")
        ),
        cited_by_count=data.get("cited_by_count") or 0,
        url=location.get("landing_page_url")
            or data.get("doi")
            or data.get("id"),
        pdf_url=best_oa.get("pdf_url")
            or location.get("pdf_url")
    )


In [9]:

def search_papers(
    query: str,
    max_results: int = 5
) -> list[Paper]:

    # Input validation
    if not isinstance(query, str):
        raise ValueError("Query must be a string.")

    if not query.strip():
        raise ValueError("Search query cannot be empty.")

    if len(query) > 500:
        raise ValueError("Search query is too long.")

    if type(max_results) is not int or not 1 <= max_results <= 20:
        raise ValueError(
            "max_results must be between 1 and 20."
        )

    url = "https://api.openalex.org/works"

    params = {
        "search": query.strip(),
        "per-page": max_results,
        "api_key": os.getenv("OPENALEX_API_KEY")
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

    except (requests.RequestException, ValueError) as error:

        raise RuntimeError(
            "OpenAlex request failed: "
            + type(error).__name__
        ) from error

    if not isinstance(data, dict):
        raise RuntimeError("Invalid API response.")

    results = data.get("results")

    if not isinstance(results, list):
        raise RuntimeError("Invalid results format.")

    papers = []
    seen_ids = set()
    seen_dois = set()

    for item in results:

        try:
            paper = parse_paper(item)

            doi = (
                paper.doi.lower().strip()
                if paper.doi else None
            )

            if paper.paper_id in seen_ids:
                continue

            if doi and doi in seen_dois:
                continue

            papers.append(paper)
            seen_ids.add(paper.paper_id)

            if doi:
                seen_dois.add(doi)

        except (ValueError, TypeError, AttributeError):
            continue

    return papers


In [10]:

query = "Adaptive learning in mathematics education"

try:

    papers = search_papers(
        query=query,
        max_results=5
    )

    print("Number of papers found:", len(papers))

    for i, paper in enumerate(papers, 1):

        print("\n" + "=" * 60)

        print(f"Paper {i}")
        print("Title:", paper.title)
        print("Authors:", ", ".join(paper.authors))
        print("Year:", paper.publication_year)
        print("DOI:", paper.doi)
        print("Citations:", paper.cited_by_count)
        print("URL:", paper.url)

except (ValueError, RuntimeError) as error:

    print("Error:", error)


Error: OpenAlex request failed: HTTPError


In [11]:

if "papers" in globals() and papers:

    df = pd.DataFrame([
        paper.model_dump()
        for paper in papers
    ])

    df["authors"] = df["authors"].apply(
        lambda authors: ", ".join(authors)
    )

    display(
        df[[
            "title",
            "authors",
            "publication_year",
            "cited_by_count",
            "doi"
        ]]
    )

else:

    print("No papers available.")


No papers available.


In [12]:

if "papers" in globals():

    for i, paper in enumerate(papers, 1):

        print(f"\nPAPER {i}")
        print("Title:", paper.title)

        print(
            "Abstract:",
            paper.abstract or "Abstract unavailable"
        )

        print("-" * 60)


In [13]:

if "papers" in globals() and papers:

    df.to_csv(
        "research_results.csv",
        index=False,
        encoding="utf-8-sig"
    )

    df.to_json(
        "research_results.json",
        orient="records",
        indent=4,
        force_ascii=False
    )

    print("Research results saved successfully.")


In [14]:
%cd /content
!git clone https://github.com/bizhamal2345/css4007-ai-agents-final.git

/content
Cloning into 'css4007-ai-agents-final'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 129 (delta 29), reused 8 (delta 8), pack-reused 75 (from 1)
Receiving objects: 100% (129/129), 59.47 KiB | 2.83 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [15]:
%cd /content/css4007-ai-agents-final

/content/css4007-ai-agents-final


In [16]:
!git pull

Already up to date.


In [17]:
!pip install -q -r requirements.txt

In [18]:
!python -m pytest tests/test_paper_search.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-9.2.0, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/css4007-ai-agents-final
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
_________________ ERROR collecting tests/test_paper_search.py __________________
/usr/local/lib/python3.13/dist-packages/_pytest/python.py:528: in importtestmodule
    mod = import_path(
/usr/local/lib/python3.13/dist-packages/_pytest/pathlib.py:640: in import_path
    importlib.import_module(module_name)
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
<frozen importlib._bootstrap>:140